In [14]:
import json
from pathlib import Path
from typing import Iterable, Union

import pandas as pd


def load_runs_into_df(run_dir: Union[str, Path], recursive: bool = True) -> pd.DataFrame:
    """Load Slay the Spire 2 .run JSON files into a metadata DataFrame.

    Args:
        run_dir: Directory containing .run files.
        recursive: Whether to search subdirectories recursively.

    Returns:
        pd.DataFrame with one row per run file and key metadata columns.
    """
    run_dir = Path(run_dir).expanduser()
    if not run_dir.exists():
        raise FileNotFoundError(f"Run directory does not exist: {run_dir}")

    pattern = "**/*.run" if recursive else "*.run"
    run_files: Iterable[Path] = sorted(run_dir.glob(pattern))

    rows = []
    for run_file in run_files:
        try:
            with run_file.open("r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as exc:
            rows.append(
                {
                    "file_path": str(run_file),
                    "file_name": run_file.name,
                    "parse_error": str(exc),
                }
            )
            continue

        players = data.get("players", []) or []
        first_player = players[0] if players else {}

        rows.append(
            {
                "file_path": str(run_file),
                "file_name": run_file.name,
                "seed": data.get("seed"),
                "start_time": data.get("start_time"),
                "run_time_seconds": data.get("run_time"),
                "win": data.get("win"),
                "was_abandoned": data.get("was_abandoned"),
                "ascension": data.get("ascension"),
                "build_id": data.get("build_id"),
                "platform_type": data.get("platform_type"),
                "schema_version": data.get("schema_version"),
                "character": first_player.get("character"),
                "deck_size": len(first_player.get("deck", []) or []),
                "relic_count": len(first_player.get("relics", []) or []),
                "killed_by_encounter": data.get("killed_by_encounter"),
                "killed_by_event": data.get("killed_by_event"),
                "parse_error": None,
            }
        )

    df = pd.DataFrame(rows)

    if not df.empty and "start_time" in df.columns:
        df["start_time_dt"] = pd.to_datetime(df["start_time"], unit="s", errors="coerce")

    return df


# Example:
df = load_runs_into_df('/Users/jzd361/Library/Application Support/SlayTheSpire2/steam/76561198187362545/profile1/saves/history')
df.head()


,file_path,file_name,seed,start_time,run_time_seconds,win,was_abandoned,ascension,build_id,platform_type,schema_version,character,deck_size,relic_count,killed_by_encounter,killed_by_event,parse_error,start_time_dt
0,/Users/jzd361/Library/Application Support/Slay...,1772746838.run,ZZHDCKMPCE,1772746838,4546,True,False,0,v0.98.0,steam,8,CHARACTER.IRONCLAD,30,16,NONE.NONE,NONE.NONE,None,2026-03-05 21:40:38
1,/Users/jzd361/Library/Application Support/Slay...,1772751643.run,P7HB90AZ0D,1772751643,3766,True,False,0,v0.98.0,steam,8,CHARACTER.SILENT,39,14,NONE.NONE,NONE.NONE,None,2026-03-05 23:00:43
2,/Users/jzd361/Library/Application Support/Slay...,1772755633.run,6UQ18BGC2W,1772755633,2113,False,False,0,v0.98.0,steam,8,CHARACTER.REGENT,18,8,ENCOUNTER.HUNTER_KILLER_NORMAL,NONE.NONE,None,2026-03-06 00:07:13
3,/Users/jzd361/Library/Application Support/Slay...,1772757850.run,22LN6LLFA9,1772757850,711,False,False,0,v0.98.0,steam,8,CHARACTER.NECROBINDER,19,4,ENCOUNTER.SKULKING_COLONY_ELITE,NONE.NONE,None,2026-03-06 00:44:10
4,/Users/jzd361/Library/Application Support/Slay...,1772758599.run,ESQ8T5SSQ8,1772758599,2911,True,False,0,v0.98.0,steam,8,CHARACTER.DEFECT,31,13,NONE.NONE,NONE.NONE,None,2026-03-06 00:56:39


In [16]:
def _safe_join(items, sep=" | "):

    return items 
    # return sep.join(str(x) for x in items) if items else ""


def _extract_act_enemies(map_point_history):
    act_enemies = {}
    for act_idx, act_nodes in enumerate(map_point_history or [], start=1):
        enemies = []
        for node in act_nodes or []:
            for room in node.get("rooms", []) or []:
                model_id = room.get("model_id")
                if model_id:
                    enemies.append(model_id)
        act_enemies[act_idx] = enemies
    return act_enemies


def load_runs_into_df(run_dir: Union[str, Path], recursive: bool = True) -> pd.DataFrame:
    """Load .run files into a metadata-rich dataframe (one row per run)."""
    run_dir = Path(run_dir).expanduser()
    if not run_dir.exists():
        raise FileNotFoundError(f"Run directory does not exist: {run_dir}")

    pattern = "**/*.run" if recursive else "*.run"
    run_files: Iterable[Path] = sorted(run_dir.glob(pattern))

    rows = []
    for run_file in run_files:
        try:
            with run_file.open("r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as exc:
            rows.append({"file_path": str(run_file), "file_name": run_file.name, "parse_error": str(exc)})
            continue

        players = data.get("players", []) or []
        first_player = players[0] if players else {}

        deck = first_player.get("deck", []) or []
        relics = first_player.get("relics", []) or []
        acts = data.get("acts", []) or []
        map_history = data.get("map_point_history", []) or []

        deck_ids = [c.get("id") for c in deck if c.get("id")]
        relic_ids = [r.get("id") for r in relics if r.get("id")]
        upgraded_cards = [c.get("id") for c in deck if c.get("id") and (c.get("current_upgrade_level", 0) or 0) > 0]

        act_enemies = _extract_act_enemies(map_history)
        all_enemies = [enemy for enemies in act_enemies.values() for enemy in enemies]

        row = {
            "file_path": str(run_file),
            "file_name": run_file.name,
            "seed": data.get("seed"),
            "start_time": data.get("start_time"),
            "run_time_seconds": data.get("run_time"),
            "win": data.get("win"),
            "was_abandoned": data.get("was_abandoned"),
            "ascension": data.get("ascension"),
            "game_mode": data.get("game_mode"),
            "build_id": data.get("build_id"),
            "platform_type": data.get("platform_type"),
            "schema_version": data.get("schema_version"),
            "character": first_player.get("character"),
            "killed_by_encounter": data.get("killed_by_encounter"),
            "killed_by_event": data.get("killed_by_event"),
            "num_acts": len(acts),
            "acts": _safe_join(acts),
            "deck_size": len(deck_ids),
            "deck_cards": _safe_join(deck_ids),
            "unique_deck_cards": len(set(deck_ids)),
            "upgraded_cards": _safe_join(upgraded_cards),
            "num_upgraded_cards": len(upgraded_cards),
            "relic_count": len(relic_ids),
            "relics": _safe_join(relic_ids),
            "unique_relics": len(set(relic_ids)),
            "enemy_count": len(all_enemies),
            "enemies_encountered": _safe_join(all_enemies),
            "unique_enemy_count": len(set(all_enemies)),
            "unique_enemies": _safe_join(sorted(set(all_enemies))),
            "parse_error": None,
        }

        for act_idx, act_name in enumerate(acts, start=1):
            enemies_this_act = act_enemies.get(act_idx, [])
            row[f"act{act_idx}_name"] = act_name
            row[f"act{act_idx}_enemy_count"] = len(enemies_this_act)
            row[f"act{act_idx}_enemies"] = _safe_join(enemies_this_act)
            row[f"act{act_idx}_unique_enemies"] = _safe_join(sorted(set(enemies_this_act)))

        rows.append(row)

    df = pd.DataFrame(rows)
    if not df.empty and "start_time" in df.columns:
        df["start_time_dt"] = pd.to_datetime(df["start_time"], unit="s", errors="coerce")
    return df


# Re-run after defining this function:
df = load_runs_into_df('/Users/jzd361/Library/Application Support/SlayTheSpire2/steam/76561198187362545/profile1/saves/history')
df.head()


,file_path,file_name,seed,start_time,run_time_seconds,win,was_abandoned,ascension,game_mode,build_id,platform_type,schema_version,character,killed_by_encounter,killed_by_event,num_acts,acts,deck_size,deck_cards,unique_deck_cards,upgraded_cards,num_upgraded_cards,relic_count,relics,unique_relics,enemy_count,enemies_encountered,unique_enemy_count,unique_enemies,parse_error,act1_name,act1_enemy_count,act1_enemies,act1_unique_enemies,act2_name,act2_enemy_count,act2_enemies,act2_unique_enemies,act3_name,act3_enemy_count,act3_enemies,act3_unique_enemies,start_time_dt
0,/Users/jzd361/Library/Application Support/Slay...,1772746838.run,ZZHDCKMPCE,1772746838,4546,True,False,0,standard,v0.98.0,steam,8,CHARACTER.IRONCLAD,NONE.NONE,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",30,"[CARD.STRIKE_IRONCLAD, CARD.STRIKE_IRONCLAD, C...",25,"[CARD.BASH, CARD.INFLAME, CARD.ARMAMENTS, CARD...",8,16,"[RELIC.BURNING_BLOOD, RELIC.BYRDPIP, RELIC.GOR...",16,33,"[ENCOUNTER.NIBBITS_WEAK, ENCOUNTER.SLIMES_WEAK...",33,"[ENCOUNTER.AXEBOTS_NORMAL, ENCOUNTER.BYRDONIS_...",None,ACT.OVERGROWTH,12,"[ENCOUNTER.NIBBITS_WEAK, ENCOUNTER.SLIMES_WEAK...","[ENCOUNTER.BYRDONIS_ELITE, ENCOUNTER.INKLETS_N...",ACT.HIVE,11,"[EVENT.TEZCATARA, ENCOUNTER.TUNNELER_WEAK, EVE...","[ENCOUNTER.DECIMILLIPEDE_ELITE, ENCOUNTER.ENTO...",ACT.GLORY,10,"[EVENT.TANX, ENCOUNTER.DEVOTED_SCULPTOR_WEAK, ...","[ENCOUNTER.AXEBOTS_NORMAL, ENCOUNTER.DEVOTED_S...",2026-03-05 21:40:38
1,/Users/jzd361/Library/Application Support/Slay...,1772751643.run,P7HB90AZ0D,1772751643,3766,True,False,0,standard,v0.98.0,steam,8,CHARACTER.SILENT,NONE.NONE,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",39,"[CARD.STRIKE_SILENT, CARD.STRIKE_SILENT, CARD....",27,"[CARD.FLICK_FLACK, CARD.RICOCHET, CARD.ACROBAT...",17,14,"[RELIC.RING_OF_THE_SNAKE, RELIC.LEAFY_POULTICE...",14,31,"[EVENT.NEOW, ENCOUNTER.SLIMES_WEAK, ENCOUNTER....",31,"[ENCOUNTER.BOWLBUGS_WEAK, ENCOUNTER.BYRDONIS_E...",None,ACT.OVERGROWTH,11,"[EVENT.NEOW, ENCOUNTER.SLIMES_WEAK, ENCOUNTER....","[ENCOUNTER.BYRDONIS_ELITE, ENCOUNTER.CEREMONIA...",ACT.HIVE,9,"[EVENT.PAEL, ENCOUNTER.EXOSKELETONS_WEAK, ENCO...","[ENCOUNTER.BOWLBUGS_WEAK, ENCOUNTER.CHOMPERS_N...",ACT.GLORY,11,"[EVENT.TANX, ENCOUNTER.TURRET_OPERATOR_WEAK, E...","[ENCOUNTER.KNIGHTS_ELITE, ENCOUNTER.MECHA_KNIG...",2026-03-05 23:00:43
2,/Users/jzd361/Library/Application Support/Slay...,1772755633.run,6UQ18BGC2W,1772755633,2113,False,False,0,standard,v0.98.0,steam,8,CHARACTER.REGENT,ENCOUNTER.HUNTER_KILLER_NORMAL,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",18,"[CARD.DEFEND_REGENT, CARD.DEFEND_REGENT, CARD....",15,"[CARD.WROUGHT_IN_WAR, CARD.REFINE_BLADE, CARD....",3,8,"[RELIC.DIVINE_RIGHT, RELIC.NEOWS_TORMENT, RELI...",8,18,"[EVENT.NEOW, ENCOUNTER.SLIMES_WEAK, ENCOUNTER....",18,"[ENCOUNTER.ENTOMANCER_ELITE, ENCOUNTER.FUZZY_W...",None,ACT.OVERGROWTH,12,"[EVENT.NEOW, ENCOUNTER.SLIMES_WEAK, ENCOUNTER....","[ENCOUNTER.FUZZY_WURM_CRAWLER_WEAK, ENCOUNTER....",ACT.HIVE,6,"[EVENT.TEZCATARA, ENCOUNTER.THIEVING_HOPPER_WE...","[ENCOUNTER.ENTOMANCER_ELITE, ENCOUNTER.HUNTER_...",ACT.GLORY,0,[],[],2026-03-06 00:07:13
3,/Users/jzd361/Library/Application Support/Slay...,1772757850.run,22LN6LLFA9,1772757850,711,False,False,0,standard,v0.98.0,steam,8,CHARACTER.NECROBINDER,ENCOUNTER.SKULKING_COLONY_ELITE,NONE.NONE,3,"[ACT.UNDERDOCKS, ACT.HIVE, ACT.GLORY]",19,"[CARD.STRIKE_NECROBINDER, CARD.STRIKE_NECROBIN...",13,[],0,4,"[RELIC.BOUND_PHYLACTERY, RELIC.SCROLL_BOXES, R...",4,9,"[EVENT.NEOW, ENCOUNTER.TOADPOLES_WEAK, EVENT.S...",9,"[ENCOUNTER.SEAPUNK_WEAK, ENCOUNTER.SKULKING_CO...",None,ACT.UNDERDOCKS,9,"[EVENT.NEOW, ENCOUNTER.TOADPOLES_WEAK, EVENT.S...","[ENCOUNTER.SEAPUNK_WEAK, ENCOUNTER.SKULKING_CO...",ACT.HIVE,0,[],[],ACT.GLORY,0,[],[],2026-03-06 00:44:10
4,/Users/jzd361/Library/Application Support/Slay...,1772758599.run,ESQ8T5SSQ8,1772758599,2911,True,False,0,standard,v0.98.0,steam,8,CHARACTER.DEFECT,NONE.NONE,NONE.NONE,3,"[ACT.UNDERDOCKS, ACT.HIVE, ACT.GLORY]",31,"[CARD.STRIKE_DEFECT, CARD.DEFEND_DEF

How many runs

In [17]:
len(df)

210

Ascensions Played

In [20]:
df.groupby("ascension").size()

ascension
0     15
1     14
2     29
3      8
4     10
5     13
6     11
7     15
8     11
9      9
10    75
dtype: int64

wins or not

In [23]:
df.groupby("win").size()

win
False    170
True      40
dtype: int64

In [24]:
40/170

0.23529411764705882

Group wins and losses by ascensions

In [25]:
df.groupby(["ascension", "win"]).size()

ascension  win  
0          False     9
           True      6
1          False     9
           True      5
2          False    24
           True      5
3          False     4
           True      4
4          False     7
           True      3
5          False    10
           True      3
6          False     8
           True      3
7          False    12
           True      3
8          False     8
           True      3
9          False     7
           True      2
10         False    72
           True      3
dtype: int64

In [26]:
df.groupby(["character", "win"]).size()

character              win  
CHARACTER.DEFECT       True       4
CHARACTER.IRONCLAD     False    110
                       True      13
CHARACTER.NECROBINDER  False     19
                       True      10
CHARACTER.REGENT       False     22
                       True       3
CHARACTER.SILENT       False     19
                       True      10
dtype: int64

Character specific winrate

In [28]:
df[df["ascension"] == 10].groupby("character").size()

character
CHARACTER.IRONCLAD       73
CHARACTER.NECROBINDER     2
dtype: int64

In [30]:
df[df["ascension"] == 10].groupby(["character", "win"]).size()

character              win  
CHARACTER.IRONCLAD     False    70
                       True      3
CHARACTER.NECROBINDER  False     2
dtype: int64

all ironclad runs

In [41]:
print(len(df[(df["character"] == "CHARACTER.IRONCLAD") & df["win"]==True]))
print(len(df[(df["character"] == "CHARACTER.IRONCLAD") & df["win"]==False]))

13
197


In [46]:
df[(df["character"] == "CHARACTER.NECROBINDER") & df["win"]==True]

,file_path,file_name,seed,start_time,run_time_seconds,win,was_abandoned,ascension,game_mode,build_id,platform_type,schema_version,character,killed_by_encounter,killed_by_event,num_acts,acts,deck_size,deck_cards,unique_deck_cards,upgraded_cards,num_upgraded_cards,relic_count,relics,unique_relics,enemy_count,enemies_encountered,unique_enemy_count,unique_enemies,parse_error,act1_name,act1_enemy_count,act1_enemies,act1_unique_enemies,act2_name,act2_enemy_count,act2_enemies,act2_unique_enemies,act3_name,act3_enemy_count,act3_enemies,act3_unique_enemies,start_time_dt
66,/Users/jzd361/Library/Application Support/Slay...,1773585832.run,DP0WWPPXVL,1773585832,3065,True,False,0,standard,v0.98.3,steam,8,CHARACTER.NECROBINDER,NONE.NONE,NONE.NONE,3,"[ACT.UNDERDOCKS, ACT.HIVE, ACT.GLORY]",29,"[CARD.BODYGUARD, CARD.UNLEASH, CARD.BONE_SHARD...",28,"[CARD.BODYGUARD, CARD.BONE_SHARDS, CARD.THE_SC...",14,15,"[RELIC.BOUND_PHYLACTERY, RELIC.LOST_COFFER, RE...",15,29,"[EVENT.NEOW, ENCOUNTER.TOADPOLES_WEAK, ENCOUNT...",29,"[ENCOUNTER.AXEBOTS_NORMAL, ENCOUNTER.BOWLBUGS_...",None,ACT.UNDERDOCKS,12,"[EVENT.NEOW, ENCOUNTER.TOADPOLES_WEAK, ENCOUNT...","[ENCOUNTER.CORPSE_SLUGS_WEAK, ENCOUNTER.CULTIS...",ACT.HIVE,8,"[EVENT.DARV, ENCOUNTER.BOWLBUGS_WEAK, ENCOUNTE...","[ENCOUNTER.BOWLBUGS_NORMAL, ENCOUNTER.BOWLBUGS...",ACT.GLORY,9,"[EVENT.NONUPEIPE, ENCOUNTER.SCROLLS_OF_BITING_...","[ENCOUNTER.AXEBOTS_NORMAL, ENCOUNTER.DEVOTED_S...",2026-03-15 14:43:52
136,/Users/jzd361/Library/Application Support/Slay...,1774621923.run,BW18EF23VN,1774621923,3068,True,False,1,standard,v0.99.1,steam,8,CHARACTER.NECROBINDER,NONE.NONE,NONE.NONE,3,"[ACT.UNDERDOCKS, ACT.HIVE, ACT.GLORY]",35,"[CARD.STRIKE_NECROBINDER, CARD.STRIKE_NECROBIN...",27,"[CARD.STRIKE_NECROBINDER, CARD.BODYGUARD, CARD...",13,19,"[RELIC.BOUND_PHYLACTERY, RELIC.GOLDEN_PEARL, R...",19,32,"[EVENT.NEOW, ENCOUNTER.SLUDGE_SPINNER_WEAK, EN...",32,"[ENCOUNTER.AXEBOTS_NORMAL, ENCOUNTER.CORPSE_SL...",None,ACT.UNDERDOCKS,10,"[EVENT.NEOW, ENCOUNTER.SLUDGE_SPINNER_WEAK, EN...","[ENCOUNTER.CORPSE_SLUGS_WEAK, ENCOUNTER.FOSSIL...",ACT.HIVE,12,"[EVENT.DARV, ENCOUNTER.THIEVING_HOPPER_WEAK, E...","[ENCOUNTER.DECIMILLIPEDE_ELITE, ENCOUNTER.INFE...",ACT.GLORY,10,"[EVENT.VAKUU, ENCOUNTER.DEVOTED_SCULPTOR_WEAK,...","[ENCOUNTER.AXEBOTS_NORMAL, ENCOUNTER.DEVOTED_S...",2026-03-27 14:32:03
137,/Users/jzd361/Library/Application Support/Slay...,1774634793.run,PTB1XYGTTV,1774634793,4781,True,False,2,standard,v0.99.1,steam,8,CHARACTER.NECROBINDER,NONE.NONE,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",20,"[CARD.UNLEASH, CARD.CLEANSE, CARD.SPUR, CARD.G...",16,"[CARD.CLEANSE, CARD.SPUR, CARD.GRAVE_WARDEN, C...",7,14,"[RELIC.BOUND_PHYLACTERY, RELIC.ARCANE_SCROLL, ...",14,32,"[EVENT.NEOW, ENCOUNTER.SLIMES_WEAK, EVENT.WELL...",32,"[ENCOUNTER.BOWLBUGS_WEAK, ENCOUNTER.CONSTRUCT_...",None,ACT.OVERGROWTH,11,"[EVENT.NEOW, ENCOUNTER.SLIMES_WEAK, EVENT.WELL...","[ENCOUNTER.FLYCONID_NORMAL, ENCOUNTER.FUZZY_WU...",ACT.HIVE,11,"[EVENT.DARV, ENCOUNTER.EXOSKELETONS_WEAK, ENCO...","[ENCOUNTER.BOWLBUGS_WEAK, ENCOUNTER.ENTOMANCER...",ACT.GLORY,10,"[EVENT.VAKUU, ENCOUNTER.DEVOTED_SCULPTOR_WEAK,...","[ENCOUNTER.CONSTRUCT_MENAGERIE_NORMAL, ENCOUNT...",2026-03-27 18:06:33
138,/Users/jzd361/Library/Application Support/Slay...,1774714234.run,10SV36UJNW,1774714234,2034,True,False,3,standard,v0.99.1,steam,8,CHARACTER.NECROBINDER,NONE.NONE,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",17,"[CARD.STRIKE_NECROBINDER, CARD.DEFEND_NECROBIN...",15,"[CARD.STRIKE_NECROBINDER, CARD.UNLEASH, CARD.S...",10,25,"[RELIC.BOUND_PHYLACTERY, RELIC.PRECARIOUS_SHEA...",25,30,"[EVENT.NEOW, ENCOUNTER.FUZZY_WURM_CRAWLER_WEAK...",29,"[ENCOUNTER.BOWLBUGS_WEAK, ENCOUNTER.BYGONE_EFF...",None,ACT.OVERGROWTH,10,"[EVENT.NEOW, ENCOUNTER.FUZZY_WURM_CRAWLER_WEAK...","[ENCOUNTER.BYGONE_EFFIGY_ELITE, ENCOUNTER.BYRD...",ACT.HIVE,10,"[EVENT.TEZCATARA, ENCOUNTER.TUNNELER_WEAK, ENC...","[ENCOUNTER.BOWLBUGS_WEAK, ENCOUNTER.DECIMILLIP...",ACT.GLORY,10,"[EVENT.VAKUU, ENCOUNTER.DEVOTED_SCULPTOR_WE

In [72]:
df[(df["character"] == "CHARACTER.IRONCLAD") & (df["win"]==False) & (df["ascension"]==10) & (df["act2_enemy_count"]==0)].head()

,file_path,file_name,seed,start_time,run_time_seconds,win,was_abandoned,ascension,game_mode,build_id,platform_type,schema_version,character,killed_by_encounter,killed_by_event,num_acts,acts,deck_size,deck_cards,unique_deck_cards,upgraded_cards,num_upgraded_cards,relic_count,relics,unique_relics,enemy_count,enemies_encountered,unique_enemy_count,unique_enemies,parse_error,act1_name,act1_enemy_count,act1_enemies,act1_unique_enemies,act2_name,act2_enemy_count,act2_enemies,act2_unique_enemies,act3_name,act3_enemy_count,act3_enemies,act3_unique_enemies,start_time_dt
77,/Users/jzd361/Library/Application Support/Slay...,1774060761.run,VAP1RG8G5H,1774060761,474,False,False,10,standard,v0.99.1,steam,8,CHARACTER.IRONCLAD,ENCOUNTER.BYRDONIS_ELITE,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",14,"[CARD.STRIKE_IRONCLAD, CARD.STRIKE_IRONCLAD, C...",9,[],0,4,"[RELIC.BURNING_BLOOD, RELIC.PRECARIOUS_SHEARS,...",4,9,"[EVENT.NEOW, ENCOUNTER.NIBBITS_WEAK, EVENT.TAB...",9,"[ENCOUNTER.BYRDONIS_ELITE, ENCOUNTER.NIBBITS_W...",None,ACT.OVERGROWTH,9,"[EVENT.NEOW, ENCOUNTER.NIBBITS_WEAK, EVENT.TAB...","[ENCOUNTER.BYRDONIS_ELITE, ENCOUNTER.NIBBITS_W...",ACT.HIVE,0,[],[],ACT.GLORY,0,[],[],2026-03-21 02:39:21
79,/Users/jzd361/Library/Application Support/Slay...,1774062945.run,QJBK3RUFZ6,1774062945,53,False,True,10,standard,v0.99.1,steam,8,CHARACTER.IRONCLAD,NONE.NONE,EVENT.NEOW,3,"[ACT.UNDERDOCKS, ACT.HIVE, ACT.GLORY]",12,"[CARD.STRIKE_IRONCLAD, CARD.STRIKE_IRONCLAD, C...",5,[],0,2,"[RELIC.BURNING_BLOOD, RELIC.CURSED_PEARL]",2,1,[EVENT.NEOW],1,[EVENT.NEOW],None,ACT.UNDERDOCKS,1,[EVENT.NEOW],[EVENT.NEOW],ACT.HIVE,0,[],[],ACT.GLORY,0,[],[],2026-03-21 03:15:45
80,/Users/jzd361/Library/Application Support/Slay...,1774063008.run,9YG9ZPP0J4,1774063008,140,False,False,10,standard,v0.99.1,steam,8,CHARACTER.IRONCLAD,ENCOUNTER.TERROR_EEL_ELITE,NONE.NONE,3,"[ACT.UNDERDOCKS, ACT.HIVE, ACT.GLORY]",14,"[CARD.STRIKE_IRONCLAD, CARD.STRIKE_IRONCLAD, C...",7,[CARD.TRUE_GRIT],1,3,"[RELIC.BURNING_BLOOD, RELIC.SILVER_CRUCIBLE, R...",3,5,"[EVENT.NEOW, ENCOUNTER.SLUDGE_SPINNER_WEAK, EV...",5,"[ENCOUNTER.SLUDGE_SPINNER_WEAK, ENCOUNTER.TERR...",None,ACT.UNDERDOCKS,5,"[EVENT.NEOW, ENCOUNTER.SLUDGE_SPINNER_WEAK, EV...","[ENCOUNTER.SLUDGE_SPINNER_WEAK, ENCOUNTER.TERR...",ACT.HIVE,0,[],[],ACT.GLORY,0,[],[],2026-03-21 03:16:48
81,/Users/jzd361/Library/Application Support/Slay...,1774063160.run,Q0H2CXGS5C,1774063160,190,False,True,10,standard,v0.99.1,steam,8,CHARACTER.IRONCLAD,ENCOUNTER.BYRDONIS_ELITE,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",16,"[CARD.STRIKE_IRONCLAD, CARD.STRIKE_IRONCLAD, C...",10,[],0,4,"[RELIC.BURNING_BLOOD, RELIC.CURSED_PEARL, RELI...",4,6,"[EVENT.NEOW, ENCOUNTER.NIBBITS_WEAK, EVENT.UNR...",6,"[ENCOUNTER.BYRDONIS_ELITE, ENCOUNTER.NIBBITS_W...",None,ACT.OVERGROWTH,6,"[EVENT.NEOW, ENCOUNTER.NIBBITS_WEAK, EVENT.UNR...","[ENCOUNTER.BYRDONIS_ELITE, ENCOUNTER.NIBBITS_W...",ACT.HIVE,0,[],[],ACT.GLORY,0,[],[],2026-03-21 03:19:20
82,/Users/jzd361/Library/Application Support/Slay...,1774063362.run,CN26B1AR0P,1774063362,267,False,False,10,standard,v0.99.1,steam,8,CHARACTER.IRONCLAD,ENCOUNTER.PHANTASMAL_GARDENERS_ELITE,NONE.NONE,3,"[ACT.UNDERDOCKS, ACT.HIVE, ACT.GLORY]",15,"[CARD.STRIKE_IRONCLAD, CARD.STRIKE_IRONCLAD, C...",8,"[CARD.RUPTURE, CARD.RUPTURE, CARD.SETUP_STRIKE]",3,2,"[RELIC.BURNING_BLOOD, RELIC.SILVER_CRUCIBLE]",2,6,"[EVENT.NEOW, ENCOUNTER.TOADPOLES_WEAK, ENCOUNT...",6,"[ENCOUNTER.CORPSE_SLUGS_WEAK, ENCOUNTER.GREMLI...",None,ACT.UNDERDOCKS,6,"[EVENT.NEOW, ENCOUNTER.TOADPOLES_WEAK, ENCOUNT...","[ENCOUNTER.CORPSE_SLUGS_WEAK, ENCOUNTER.GREMLI...",ACT.HIVE,0,[],[],ACT.GLORY,0,[],[],2026-03-21 03:22:42


In [69]:
df[(df["character"] == "CHARACTER.IRONCLAD") & (df["win"]==False) & (df["ascension"]==10) & (df["act3_enemy_count"]==0) & (df["act2_enemy_count"]>0)].groupby("killed_by_encounter").size().sort_values(ascending=False)

killed_by_encounter
ENCOUNTER.KNOWLEDGE_DEMON_BOSS        3
ENCOUNTER.DECIMILLIPEDE_ELITE         2
ENCOUNTER.ENTOMANCER_ELITE            2
ENCOUNTER.KAISER_CRAB_BOSS            2
ENCOUNTER.HUNTER_KILLER_NORMAL        1
ENCOUNTER.INFESTED_PRISMS_ELITE       1
ENCOUNTER.SLUMBERING_BEETLE_NORMAL    1
ENCOUNTER.THE_OBSCURA_NORMAL          1
ENCOUNTER.THIEVING_HOPPER_WEAK        1
dtype: int64

In [71]:
df[(df["character"] == "CHARACTER.IRONCLAD") & (df["win"]==False) & (df["act3_enemy_count"]==0) & (df["act2_enemy_count"]>0)].groupby("killed_by_encounter").size().sort_values(ascending=False)

killed_by_encounter
ENCOUNTER.DECIMILLIPEDE_ELITE         6
ENCOUNTER.KNOWLEDGE_DEMON_BOSS        4
ENCOUNTER.KAISER_CRAB_BOSS            3
ENCOUNTER.BOWLBUGS_NORMAL             2
ENCOUNTER.ENTOMANCER_ELITE            2
ENCOUNTER.INFESTED_PRISMS_ELITE       2
ENCOUNTER.THE_INSATIABLE_BOSS         2
ENCOUNTER.HUNTER_KILLER_NORMAL        1
ENCOUNTER.OVICOPTER_NORMAL            1
ENCOUNTER.SLUMBERING_BEETLE_NORMAL    1
ENCOUNTER.THE_OBSCURA_NORMAL          1
ENCOUNTER.THIEVING_HOPPER_WEAK        1
ENCOUNTER.TUNNELER_NORMAL             1
dtype: int64

In [ ]:
df[(df["character"] == "CHARACTER.IRONCLAD") & (df["win"]==False) & 

,file_path,file_name,seed,start_time,run_time_seconds,win,was_abandoned,ascension,game_mode,build_id,platform_type,schema_version,character,killed_by_encounter,killed_by_event,num_acts,acts,deck_size,deck_cards,unique_deck_cards,upgraded_cards,num_upgraded_cards,relic_count,relics,unique_relics,enemy_count,enemies_encountered,unique_enemy_count,unique_enemies,parse_error,act1_name,act1_enemy_count,act1_enemies,act1_unique_enemies,act2_name,act2_enemy_count,act2_enemies,act2_unique_enemies,act3_name,act3_enemy_count,act3_enemies,act3_unique_enemies,start_time_dt
78,/Users/jzd361/Library/Application Support/Slay...,1774061246.run,S7YU47G6AC,1774061246,1685,False,False,10,standard,v0.99.1,steam,8,CHARACTER.IRONCLAD,ENCOUNTER.KNOWLEDGE_DEMON_BOSS,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",32,"[CARD.STRIKE_IRONCLAD, CARD.STRIKE_IRONCLAD, C...",23,"[CARD.UNMOVABLE, CARD.STOKE, CARD.POMMEL_STRIK...",5,19,"[RELIC.BURNING_BLOOD, RELIC.LARGE_CAPSULE, REL...",19,22,"[EVENT.NEOW, ENCOUNTER.NIBBITS_WEAK, ENCOUNTER...",22,"[ENCOUNTER.BYGONE_EFFIGY_ELITE, ENCOUNTER.BYRD...",None,ACT.OVERGROWTH,12,"[EVENT.NEOW, ENCOUNTER.NIBBITS_WEAK, ENCOUNTER...","[ENCOUNTER.BYGONE_EFFIGY_ELITE, ENCOUNTER.BYRD...",ACT.HIVE,10,"[EVENT.TEZCATARA, ENCOUNTER.EXOSKELETONS_WEAK,...","[ENCOUNTER.CHOMPERS_NORMAL, ENCOUNTER.DECIMILL...",ACT.GLORY,0,[],[],2026-03-21 02:47:26
83,/Users/jzd361/Library/Application Support/Slay...,1774063641.run,LSBAA82LCX,1774063641,936,False,False,10,standard,v0.99.1,steam,8,CHARACTER.IRONCLAD,ENCOUNTER.DECIMILLIPEDE_ELITE,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",16,"[CARD.STRIKE_IRONCLAD, CARD.DEFEND_IRONCLAD, C...",13,"[CARD.INFERNO, CARD.TAUNT]",2,7,"[RELIC.BURNING_BLOOD, RELIC.PRECARIOUS_SHEARS,...",7,16,"[EVENT.NEOW, ENCOUNTER.NIBBITS_WEAK, ENCOUNTER...",16,"[ENCOUNTER.DECIMILLIPEDE_ELITE, ENCOUNTER.EXOS...",None,ACT.OVERGROWTH,10,"[EVENT.NEOW, ENCOUNTER.NIBBITS_WEAK, ENCOUNTER...","[ENCOUNTER.FUZZY_WURM_CRAWLER_WEAK, ENCOUNTER....",ACT.HIVE,6,"[EVENT.OROBAS, ENCOUNTER.THIEVING_HOPPER_WEAK,...","[ENCOUNTER.DECIMILLIPEDE_ELITE, ENCOUNTER.EXOS...",ACT.GLORY,0,[],[],2026-03-21 03:27:21
87,/Users/jzd361/Library/Application Support/Slay...,1774105704.run,3QMPUNUC2H,1774105704,1317,False,False,10,standard,v0.99.1,steam,8,CHARACTER.IRONCLAD,ENCOUNTER.THE_OBSCURA_NORMAL,NONE.NONE,3,"[ACT.UNDERDOCKS, ACT.HIVE, ACT.GLORY]",19,"[CARD.STRIKE_IRONCLAD, CARD.STRIKE_IRONCLAD, C...",12,"[CARD.STRIKE_IRONCLAD, CARD.TWIN_STRIKE, CARD....",8,12,"[RELIC.BURNING_BLOOD, RELIC.LAVA_ROCK, RELIC.J...",12,20,"[EVENT.NEOW, ENCOUNTER.TOADPOLES_WEAK, EVENT.D...",20,"[ENCOUNTER.CHOMPERS_NORMAL, ENCOUNTER.CORPSE_S...",None,ACT.UNDERDOCKS,12,"[EVENT.NEOW, ENCOUNTER.TOADPOLES_WEAK, EVENT.D...","[ENCOUNTER.CORPSE_SLUGS_WEAK, ENCOUNTER.FOSSIL...",ACT.HIVE,8,"[EVENT.PAEL, ENCOUNTER.EXOSKELETONS_WEAK, ENCO...","[ENCOUNTER.CHOMPERS_NORMAL, ENCOUNTER.DECIMILL...",ACT.GLORY,0,[],[],2026-03-21 15:08:24
95,/Users/jzd361/Library/Application Support/Slay...,1774150472.run,KD4D13BW44,1774150472,1123,False,True,10,standard,v0.100.0,steam,8,CHARACTER.IRONCLAD,ENCOUNTER.ENTOMANCER_ELITE,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",22,"[CARD.STRIKE_IRONCLAD, CARD.STRIKE_IRONCLAD, C...",15,[CARD.STAMPEDE],1,7,"[RELIC.BURNING_BLOOD, RELIC.NEW_LEAF, RELIC.PA...",7,22,"[EVENT.NEOW, ENCOUNTER.SHRINKER_BEETLE_WEAK, E...",22,"[ENCOUNTER.BOWLBUGS_NORMAL, ENCOUNTER.BOWLBUGS...",None,ACT.OVERGROWTH,11,"[EVENT.NEOW, ENCOUNTER.SHRINKER_BEETLE_WEAK, E...","[ENCOUNTER.CEREMONIAL_BEAST_BOSS, ENCOUNTER.CU...",ACT.HIVE,11,"[EVENT.OROBAS, ENCOUNTER.BOWLBUGS_WEAK, EVENT....","[ENCOUNTER.BOWLBUGS_NORMAL, ENCOUNTER.BOWLBUGS...",ACT.GLORY,0,[],[],2026-03-22 03:34:32
98,/Users/jzd361/Library/Application Support/Slay...,1774212966.run,DWTJHBX3QK,1774212966,1633,False,False,10,standard,v0.100.0,steam,8,CHARACTER.IRONCLAD,ENCOUNTER.HUNTER_KILLER_NORMAL,NONE.NONE,3,"[ACT.UNDERDOCKS, ACT.HIVE, ACT.GLORY]",23,"[CARD.STRIKE_IRONCLAD, CARD.STRIKE_IRONCLAD, C...",16,[CARD.BRIGHTEST_F

In [55]:
df[(df["character"] == "CHARACTER.IRONCLAD") & (df["win"]==False) & (df["ascension"]==10) & (df["act2_enemy_count"]>0)].groupby("killed_by_encounter").size().sort_values(ascending=False)

killed_by_encounter
ENCOUNTER.KNOWLEDGE_DEMON_BOSS        3
ENCOUNTER.DECIMILLIPEDE_ELITE         2
ENCOUNTER.ENTOMANCER_ELITE            2
ENCOUNTER.KAISER_CRAB_BOSS            2
ENCOUNTER.QUEEN_BOSS                  2
ENCOUNTER.HUNTER_KILLER_NORMAL        1
ENCOUNTER.INFESTED_PRISMS_ELITE       1
ENCOUNTER.SLUMBERING_BEETLE_NORMAL    1
ENCOUNTER.SOUL_NEXUS_ELITE            1
ENCOUNTER.TEST_SUBJECT_BOSS           1
ENCOUNTER.THE_OBSCURA_NORMAL          1
ENCOUNTER.THIEVING_HOPPER_WEAK        1
dtype: int64

In [53]:
df[(df["character"] == "CHARACTER.IRONCLAD") & (df["win"]==False) & (df["ascension"]==10) & (df["act3_enemy_count"]>0)]

,file_path,file_name,seed,start_time,run_time_seconds,win,was_abandoned,ascension,game_mode,build_id,platform_type,schema_version,character,killed_by_encounter,killed_by_event,num_acts,acts,deck_size,deck_cards,unique_deck_cards,upgraded_cards,num_upgraded_cards,relic_count,relics,unique_relics,enemy_count,enemies_encountered,unique_enemy_count,unique_enemies,parse_error,act1_name,act1_enemy_count,act1_enemies,act1_unique_enemies,act2_name,act2_enemy_count,act2_enemies,act2_unique_enemies,act3_name,act3_enemy_count,act3_enemies,act3_unique_enemies,start_time_dt
99,/Users/jzd361/Library/Application Support/Slay...,1774214671.run,DY1S5TWZX5,1774214671,2043,False,False,10,standard,v0.100.0,steam,8,CHARACTER.IRONCLAD,ENCOUNTER.SOUL_NEXUS_ELITE,NONE.NONE,3,"[ACT.UNDERDOCKS, ACT.HIVE, ACT.GLORY]",20,"[CARD.STRIKE_IRONCLAD, CARD.DEFEND_IRONCLAD, C...",19,"[CARD.BASH, CARD.DISMANTLE, CARD.BATTLE_TRANCE...",12,14,"[RELIC.BURNING_BLOOD, RELIC.PRECARIOUS_SHEARS,...",14,25,"[EVENT.NEOW, ENCOUNTER.SEAPUNK_WEAK, ENCOUNTER...",25,"[ENCOUNTER.BOWLBUGS_NORMAL, ENCOUNTER.DEVOTED_...",None,ACT.UNDERDOCKS,11,"[EVENT.NEOW, ENCOUNTER.SEAPUNK_WEAK, ENCOUNTER...","[ENCOUNTER.GREMLIN_MERC_NORMAL, ENCOUNTER.HAUN...",ACT.HIVE,9,"[EVENT.TEZCATARA, ENCOUNTER.EXOSKELETONS_WEAK,...","[ENCOUNTER.BOWLBUGS_NORMAL, ENCOUNTER.ENTOMANC...",ACT.GLORY,5,"[EVENT.NONUPEIPE, ENCOUNTER.SCROLLS_OF_BITING_...","[ENCOUNTER.DEVOTED_SCULPTOR_WEAK, ENCOUNTER.SC...",2026-03-22 21:24:31
106,/Users/jzd361/Library/Application Support/Slay...,1774296612.run,YS6JB4W171,1774296612,2822,False,False,10,standard,v0.99.1,steam,8,CHARACTER.IRONCLAD,ENCOUNTER.QUEEN_BOSS,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",22,"[CARD.STRIKE_IRONCLAD, CARD.STRIKE_IRONCLAD, C...",19,"[CARD.STRIKE_IRONCLAD, CARD.STRIKE_IRONCLAD, C...",8,17,"[RELIC.BURNING_BLOOD, RELIC.CURSED_PEARL, RELI...",17,36,"[EVENT.NEOW, ENCOUNTER.SLIMES_WEAK, EVENT.JUNG...",36,"[ENCOUNTER.BOWLBUGS_NORMAL, ENCOUNTER.BOWLBUGS...",None,ACT.OVERGROWTH,13,"[EVENT.NEOW, ENCOUNTER.SLIMES_WEAK, EVENT.JUNG...","[ENCOUNTER.BYGONE_EFFIGY_ELITE, ENCOUNTER.CERE...",ACT.HIVE,12,"[EVENT.PAEL, ENCOUNTER.TUNNELER_WEAK, ENCOUNTE...","[ENCOUNTER.BOWLBUGS_NORMAL, ENCOUNTER.BOWLBUGS...",ACT.GLORY,11,"[EVENT.NONUPEIPE, ENCOUNTER.SCROLLS_OF_BITING_...","[ENCOUNTER.DOORMAKER_BOSS, ENCOUNTER.FROG_KNIG...",2026-03-23 20:10:12
162,/Users/jzd361/Library/Application Support/Slay...,1775272456.run,89GWN6H3HT,1775272456,1808,False,False,10,standard,v0.99.1,steam,8,CHARACTER.IRONCLAD,ENCOUNTER.TEST_SUBJECT_BOSS,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",10,"[CARD.ASCENDERS_BANE, CARD.POMMEL_STRIKE, CARD...",8,"[CARD.SECOND_WIND, CARD.EXPECT_A_FIGHT, CARD.O...",7,15,"[RELIC.BURNING_BLOOD, RELIC.PRECISE_SCISSORS, ...",15,34,"[EVENT.NEOW, ENCOUNTER.SLIMES_WEAK, ENCOUNTER....",34,"[ENCOUNTER.CUBEX_CONSTRUCT_NORMAL, ENCOUNTER.E...",None,ACT.OVERGROWTH,12,"[EVENT.NEOW, ENCOUNTER.SLIMES_WEAK, ENCOUNTER....","[ENCOUNTER.CUBEX_CONSTRUCT_NORMAL, ENCOUNTER.I...",ACT.HIVE,11,"[EVENT.TEZCATARA, ENCOUNTER.EXOSKELETONS_WEAK,...","[ENCOUNTER.ENTOMANCER_ELITE, ENCOUNTER.EXOSKEL...",ACT.GLORY,11,"[EVENT.TANX, ENCOUNTER.TURRET_OPERATOR_WEAK, E...","[ENCOUNTER.KNIGHTS_ELITE, ENCOUNTER.MECHA_KNIG...",2026-04-04 03:14:16
181,/Users/jzd361/Library/Application Support/Slay...,1775833823.run,QP950MW7QV,1775833823,2126,False,False,10,custom,v0.103.0,steam,9,CHARACTER.IRONCLAD,ENCOUNTER.QUEEN_BOSS,NONE.NONE,3,"[ACT.OVERGROWTH, ACT.HIVE, ACT.GLORY]",182,"[CARD.ASCENDERS_BANE, CARD.INJURY, CARD.INJURY...",55,"[CARD.BRAND, CARD.BULLY]",2,46,"[RELIC.BURNING_BLOOD, RELIC.VAMBRACE, RELIC.RE...",46,35,"[EVENT.NEOW, ENCOUNTER.SHRINKER_BEETLE_WEAK, E...",30,"[ENCOUNTER.BOWLBUGS_WEAK, ENCOUNTER.BYGONE_EFF...",None,ACT.OVERGROWTH,14,"[EVENT.NEOW, ENCOUNTER.SHRINKER_BEETLE_WEAK, E...","[ENCOUNTER.BYGONE_EFFIGY_ELITE, ENCOUNTER.BYRD...",ACT.HIVE,11,"[EVENT.OROBAS, ENCOUNTER.BOWLBUGS_WEAK, ENCOUN...","[ENCOUNTER.BOWLBUGS_WEAK, ENCOUNTER.DECIMILLIP...",ACT.GLORY,10,"[EVENT.TANX, ENCO

In [ ]:
# file_path
# file_name
# seed
# start_time
# run_time_seconds
# win
# was_abandoned
# ascension
# game_mode
# build_id
# platform_type
# schema_version
# character
# killed_by_encounter
# killed_by_event
# num_acts
# acts
# deck_size
# deck_cards
# unique_deck_cards
# upgraded_cards
# num_upgraded_cards
# relic_count
# relics
# unique_relics
# enemy_count
# enemies_encountered
# unique_enemy_count
# unique_enemies
# parse_error
# act1_name
# act1_enemy_count
# act1_enemies
# act1_unique_enemies
# act2_name
# act2_enemy_count
# act2_enemies
# act2_unique_enemies
# act3_name
# act3_enemy_count
# act3_enemies
# act3_unique_enemies
# start_time_dt